# World Commodities (Yahoo Finance)

## Setup

In [1]:
# Standard library
from pathlib import Path
from typing import cast

import mgplot as mg

# Third party
import pandas as pd
import requests
import yfinance as yf

In [2]:
# Pandas display
pd.options.display.max_rows = 999999

# Chart output directory
CHART_DIR = "./CHARTS/Commodities/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

# Plotting control
SHOW = False
SOURCE_YAHOO = "Source: Yahoo Finance"

# Date-range starts
SHORT_START = "2026-01-01"   # ~3-month window for oil and gas benchmarks
LONG_START = "2024-03-20"    # ~2-year window for metals, grains, softs, indices

# Shared event annotation
HORMUZ_CLOSURE = {
    "x": pd.Period("2026-02-28", freq="D").ordinal,
    "color": "grey",
    "linestyle": "--",
    "linewidth": 1,
    "label": "Hormuz closure",
}

# Oilprice.com source cache
CACHE_DIR = Path("./CACHE/YAHOO_oilprice/")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OILPRICE_BLENDS = {
    "WTI (US)": {"blend_id": 45, "cache": "wti.csv"},
    "Brent (Europe)": {"blend_id": 46, "cache": "brent.csv"},
    "Oman (Middle East)": {"blend_id": 48, "cache": "dme_oman.csv"},
    "Dubai (Middle East)": {"blend_id": 144, "cache": "dubai.csv"},
}

# CME dated-futures codes used to build forward-curve symbols
MONTH_CODES = "FGHJKMNQUVXZ"
N_FORWARD_MONTHS = 18

# Natural gas unit conversion (TTF quoted in EUR per MWh)
MMBTU_PER_MWH = 3.412

# Singapore Mogas 92 unit conversion (MPS quoted in USD per metric tonne)
MOGAS_BBL_PER_TONNE = 8.33

## Helper functions

In [3]:
def fetch_yahoo_close(ticker: str, start: str) -> pd.Series:
    """Fetch daily closing prices for a Yahoo ticker as a PeriodIndex series."""
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        return pd.Series(dtype=float)
    s = raw["Close"].squeeze().dropna()
    s.index = cast("pd.DatetimeIndex", s.index).to_period("D")
    return s


def fetch_yahoo_frame(tickers: dict[str, str], start: str) -> pd.DataFrame:
    """Fetch closing prices for multiple tickers into a single DataFrame."""
    frames = {label: fetch_yahoo_close(ticker, start) for ticker, label in tickers.items()}
    return pd.DataFrame(frames)


def summarise(df: pd.DataFrame, label: str) -> None:
    """Print date range and min/max summary for each column."""
    valid = df.dropna(how="all")
    print(f"{label}: {valid.index[0]} to {valid.index[-1]}")
    for col in df.columns:
        s = df[col].dropna()
        if len(s):
            print(f"  {col:22s}  n={len(s):3d}  min={s.min():.2f}  max={s.max():.2f}")


def latest_date_str(df: pd.DataFrame) -> str:
    """Return the last non-NaN date as a formatted string."""
    return df.dropna(how="all").index[-1].strftime("%-d-%b-%Y")

In [4]:
def plot_frame(
    df: pd.DataFrame,
    *,
    title: str,
    ylabel: str,
    lfooter: str,
    rfooter: str = SOURCE_YAHOO,
    axvline: dict | None = None,
) -> None:
    """Plot a multi-series DataFrame with standard chart formatting."""
    mg.line_plot_finalise(
        df,
        title=title,
        ylabel=ylabel,
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        axvline=axvline,
        lfooter=lfooter,
        rfooter=rfooter,
        show=SHOW,
    )


def plot_single(
    series: pd.Series, *, name: str, ylabel: str, is_futures: bool = True
) -> None:
    """Plot a single commodity series with standard formatting."""
    data_to = series.index[-1].strftime("%-d-%b-%Y")
    if is_futures:
        title = f"{name} Futures Price"
        footer = f"Front-month futures. Data to {data_to}."
    else:
        title = name
        footer = f"Daily close. Data to {data_to}."
    mg.line_plot_finalise(
        series,
        title=title,
        ylabel=ylabel,
        xlabel=None,
        annotate=True,
        lfooter=footer,
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


def chart_single_tickers(
    tickers: list[tuple[str, str, str]], start: str, is_futures: bool = True
) -> None:
    """Fetch and chart each (ticker, name, ylabel) tuple independently."""
    for ticker, name, ylabel in tickers:
        s = fetch_yahoo_close(ticker, start)
        if len(s) == 0:
            print(f"{name} ({ticker}): no data, skipping")
            continue
        print(f"{name}: {s.index[0]} to {s.index[-1]}  min={s.min():.2f}  max={s.max():.2f}")
        plot_single(s, name=name, ylabel=ylabel, is_futures=is_futures)

In [5]:
def _contract_months(n_months: int) -> list[pd.Period]:
    start = pd.Period(pd.Timestamp.today(), freq="M") + 1
    return [start + i for i in range(n_months)]


def _contract_symbol(root: str, period: pd.Period) -> str:
    code = MONTH_CODES[period.month - 1]
    yy = period.year % 100
    return f"{root}{code}{yy:02d}.NYM"


def fetch_forward_curve(root: str, n_months: int) -> pd.DataFrame:
    """Fetch latest close + trade date for each dated contract."""
    records = {}
    for m in _contract_months(n_months):
        sym = _contract_symbol(root, m)
        try:
            hist = yf.Ticker(sym).history(period="5d")
        except (KeyError, ValueError, OSError):
            continue
        if hist is None or len(hist) == 0:
            continue
        records[m] = {
            "price": float(hist["Close"].iloc[-1]),
            "date": hist.index[-1].tz_localize(None).normalize(),
        }
    return pd.DataFrame.from_dict(records, orient="index").sort_index()


def build_forward_curves(n_months: int) -> tuple[pd.DataFrame, pd.Timestamp]:
    """Fetch WTI and Brent forward curves and return prices with trade date."""
    wti = fetch_forward_curve("CL", n_months)
    brent = fetch_forward_curve("BZ", n_months)
    prices = pd.DataFrame({
        "WTI (US)": wti["price"],
        "Brent (Europe)": brent["price"],
    })
    prices.index = pd.PeriodIndex(prices.index, freq="M")
    latest = max(wti["date"].max(), brent["date"].max())
    return prices, latest


def plot_forward_curves(df: pd.DataFrame, as_of: pd.Timestamp) -> None:
    """Chart WTI and Brent forward curves by contract month."""
    as_of_str = as_of.strftime("%-d-%b-%Y")
    mg.line_plot_finalise(
        df,
        title="Crude Oil Forward Curves: WTI and Brent",
        ylabel="USD per barrel",
        xlabel="Contract month",
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        marker="o",
        markersize=4,
        lfooter=(
            f"Latest settle price for each dated contract (delivery within month). "
            f"As of {as_of_str}."
        ),
        rfooter="Source: Yahoo Finance (NYMEX CL, ICE BZ)",
        show=SHOW,
    )

In [6]:
def _fetch_oilprice(blend_id: int, period: int) -> pd.DataFrame:
    """Fetch crude oil prices from Oilprice.com's public JSON endpoint."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://oilprice.com/oil-price-charts/",
    }
    csrf = requests.get(
        "https://oilprice.com/ajax/csrf", headers=headers, timeout=15
    ).json()
    resp = requests.post(
        "https://oilprice.com/freewidgets/json_get_oilprices",
        headers=headers,
        data={"blend_id": blend_id, "period": period, csrf["name"]: csrf["hash"]},
        timeout=15,
    )
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["prices"])
    df["date"] = pd.to_datetime(df["time"], unit="s").dt.normalize()
    df["price"] = df["price"].astype(float)
    df = df.sort_values("date").reset_index(drop=True)
    # Drop stale artifact entries (timestamps far out of sequence)
    if len(df) > 1:
        df = df[df["date"] >= df["date"].iloc[-1] - pd.Timedelta(days=400)]
    return df[["date", "price"]]


def load_oilprice_blends() -> dict[str, pd.Series]:
    """Load all configured Oilprice.com blends, refreshing the on-disk cache."""
    result: dict[str, pd.Series] = {}
    for name, cfg in OILPRICE_BLENDS.items():
        cache_file = CACHE_DIR / cfg["cache"]
        if cache_file.exists():
            cached = pd.read_csv(cache_file, parse_dates=["date"])
            fresh = _fetch_oilprice(cfg["blend_id"], period=6)
            combined = pd.concat([cached, fresh]).drop_duplicates(subset="date", keep="last")
            df = combined.sort_values("date").reset_index(drop=True)
            print(f"{name}: updated cache ({len(cached)} -> {len(df)} rows)")
        else:
            df = _fetch_oilprice(cfg["blend_id"], period=5)
            print(f"{name}: initial cache ({len(df)} rows)")
        df.to_csv(cache_file, index=False)
        idx = pd.PeriodIndex(df["date"], freq="D")
        s = pd.Series(df["price"].values, index=idx, name=name)
        # Guard against duplicate dates in source data
        s = s[~s.index.duplicated(keep="last")]
        result[name] = s
    return result

## Crude oil: spot prices

In [7]:
def run_crude_spot() -> pd.DataFrame:
    """Fetch and chart crude oil spot prices from Oilprice.com."""
    spot_series = load_oilprice_blends()
    prices = pd.DataFrame(spot_series)
    prices = prices.sort_index()
    prices = prices.loc[prices.index >= pd.Period(SHORT_START, freq="D")]

    summarise(prices, "Crude spot")

    # Per-series last dates for footer
    last_dates = {
        col: prices[col].dropna().index[-1].strftime("%-d-%b")
        for col in prices.columns
    }
    date_str = ", ".join(f"{col.split(' ')[0]} {d}" for col, d in last_dates.items())
    lfooter = f"Daily spot prices. Last: {date_str}."

    mg.line_plot_finalise(
        prices,
        title="Crude Oil Spot Prices",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=lfooter,
        rfooter="Source: Oilprice.com",
        show=SHOW,
    )
    return prices


spot_prices = run_crude_spot()

WTI (US): updated cache (307 -> 307 rows)


Brent (Europe): updated cache (308 -> 308 rows)


Oman (Middle East): updated cache (206 -> 206 rows)


Dubai (Middle East): updated cache (217 -> 217 rows)
Crude spot: 2026-01-01 to 2026-04-17
  WTI (US)                n= 92  min=55.99  max=112.95
  Brent (Europe)          n= 92  min=59.96  max=109.77
  Oman (Middle East)      n= 58  min=58.36  max=162.72
  Dubai (Middle East)     n= 56  min=58.35  max=137.82


## Crude oil: front-month futures

In [8]:
def run_crude_futures() -> pd.DataFrame:
    """Fetch and chart WTI vs Brent front-month futures."""
    df = fetch_yahoo_frame(
        {"CL=F": "WTI (US)", "BZ=F": "Brent (Europe)"},
        start=SHORT_START,
    )
    summarise(df, "Crude futures")
    mg.line_plot_finalise(
        df,
        title="Crude Oil Futures: WTI vs Brent",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Front-month futures. NYMEX (WTI) and ICE (Brent). Data to {latest_date_str(df)}.",
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )
    return df


wti_brent = run_crude_futures()

Crude futures: 2026-01-02 to 2026-04-17
  WTI (US)                n= 73  min=55.99  max=112.95
  Brent (Europe)          n= 73  min=59.96  max=118.35


## Crude oil: forward curves

In [9]:
def run_forward_curves() -> None:
    """Fetch and chart crude oil forward curves."""
    forward, as_of = build_forward_curves(N_FORWARD_MONTHS)
    print(f"As of {as_of.date()}")
    print(f"Contracts: {forward.index[0]} to {forward.index[-1]}")
    for col in forward.columns:
        valid = forward[col].dropna()
        print(f"  {col:20s}  n={len(valid)}  front={valid.iloc[0]:.2f}  back={valid.iloc[-1]:.2f}")
    plot_forward_curves(forward, as_of)


run_forward_curves()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BZK26.NYM"}}}


$BZK26.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BZQ27.NYM"}}}


$BZQ27.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


$BZU27.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


$BZV27.NYM: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


As of 2026-04-17
Contracts: 2026-05 to 2027-10
  WTI (US)              n=18  front=85.57  back=69.42
  Brent (Europe)        n=14  front=91.87  back=78.13


## Crude oil: spot vs front-month futures

In [10]:
def run_spot_vs_futures(spot_prices: pd.DataFrame, futures_prices: pd.DataFrame) -> None:
    """Chart spot vs front-month futures for WTI and Brent."""
    # WTI comparison
    wti = pd.DataFrame({
        "Spot": spot_prices["WTI (US)"],
        "Front-month futures": futures_prices["WTI (US)"],
    })
    summarise(wti, "WTI spot vs futures")
    mg.line_plot_finalise(
        wti,
        title="WTI Crude Oil: Spot vs Futures",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Spot: Oilprice.com. Futures: NYMEX CL front month. Data to {latest_date_str(wti)}.",
        rfooter="Source: Oilprice.com / Yahoo Finance",
        show=SHOW,
    )

    # Brent comparison
    brent = pd.DataFrame({
        "Spot": spot_prices["Brent (Europe)"],
        "Front-month futures": futures_prices["Brent (Europe)"],
    })
    summarise(brent, "Brent spot vs futures")
    mg.line_plot_finalise(
        brent,
        title="Brent Crude Oil: Spot vs Futures",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Spot: Oilprice.com. Futures: ICE BZ front month. Data to {latest_date_str(brent)}.",
        rfooter="Source: Oilprice.com / Yahoo Finance",
        show=SHOW,
    )


run_spot_vs_futures(spot_prices, wti_brent)

WTI spot vs futures: 2026-01-01 to 2026-04-17
  Spot                    n= 92  min=55.99  max=112.95
  Front-month futures     n= 73  min=55.99  max=112.95
Brent spot vs futures: 2026-01-01 to 2026-04-17
  Spot                    n= 92  min=59.96  max=109.77
  Front-month futures     n= 73  min=59.96  max=118.35


## Singapore refined products: gasoil and petrol

In [11]:
def run_singapore_products() -> None:
    """Fetch and chart Singapore Gasoil and Mogas 92 front-month futures."""
    gasoil = fetch_yahoo_close("SGB=F", start=SHORT_START)             # USD/bbl
    mogas_per_tonne = fetch_yahoo_close("MPS=F", start=SHORT_START)    # USD/tonne
    mogas_per_bbl = mogas_per_tonne / MOGAS_BBL_PER_TONNE
    df = pd.DataFrame({
        "Gasoil 10ppm (diesel)": gasoil,
        "Mogas 92 (petrol)": mogas_per_bbl,
    })
    summarise(df, "Singapore products")
    mg.line_plot_finalise(
        df,
        title="Singapore Refined Product Futures: Gasoil and Petrol",
        ylabel="USD per barrel",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        rheader="Refined products (not crude)",
        lfooter=(
            f"NYMEX front-month futures on Platts Singapore Gasoil (diesel) "
            f"and Mogas 92 (petrol). Data to {latest_date_str(df)}."
        ),
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


run_singapore_products()

Singapore products: 2026-01-02 to 2026-04-16
  Gasoil 10ppm (diesel)   n= 72  min=77.68  max=232.57
  Mogas 92 (petrol)       n= 72  min=54.88  max=102.63


## Precious and base metals

In [12]:
METALS = [
    ("GC=F", "Gold", "USD per troy ounce"),
    ("SI=F", "Silver", "USD per troy ounce"),
    ("PL=F", "Platinum", "USD per troy ounce"),
    ("PA=F", "Palladium", "USD per troy ounce"),
    ("HG=F", "Copper", "USD per pound"),
    ("TIO=F", "Iron Ore", "USD per tonne"),
]

chart_single_tickers(METALS, start=LONG_START)

Gold: 2024-03-20 to 2026-04-17  min=2157.90  max=5318.40


Silver: 2024-03-20 to 2026-04-17  min=24.48  max=115.08


Platinum: 2024-03-20 to 2026-04-17  min=894.00  max=2852.40


Palladium: 2024-03-20 to 2026-04-17  min=822.80  max=2169.90


Copper: 2024-03-20 to 2026-04-17  min=3.94  max=6.18


Iron Ore: 2024-03-20 to 2026-04-16  min=91.28  max=119.56


## Natural gas benchmarks

In [13]:
def build_gas_prices() -> pd.DataFrame:
    """Fetch gas benchmarks and convert TTF to USD/MMBtu."""
    gas = fetch_yahoo_frame(
        {
            "NG=F": "Henry Hub (US)",
            "TTF=F": "TTF (Europe)",
            "JKM=F": "JKM (Asia)",
        },
        start=SHORT_START,
    ).dropna()
    eurusd = fetch_yahoo_close("EURUSD=X", start=SHORT_START)
    # Convert TTF from EUR/MWh to USD/MMBtu
    ttf_aligned = eurusd.reindex(gas.index, method="ffill")
    gas["TTF (Europe)"] = gas["TTF (Europe)"] * ttf_aligned / MMBTU_PER_MWH
    return gas


def run_natural_gas() -> None:
    """Fetch and chart natural gas benchmarks."""
    gas = build_gas_prices()
    summarise(gas, "Natural gas")
    mg.line_plot_finalise(
        gas,
        title="Natural Gas Futures Benchmarks",
        ylabel="USD per MMBtu",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        rounding=2,
        axvline=HORMUZ_CLOSURE,
        lfooter=f"Front-month futures. TTF converted from EUR/MWh using daily EUR/USD rate. Data to {latest_date_str(gas)}.",
        rfooter=SOURCE_YAHOO,
        show=SHOW,
    )


run_natural_gas()

Natural gas: 2026-01-02 to 2026-04-16
  Henry Hub (US)          n= 72  min=2.60  max=7.46
  TTF (Europe)            n= 72  min=9.40  max=20.78
  JKM (Asia)              n= 72  min=9.54  max=22.35


## Grains

In [14]:
def run_grains() -> None:
    """Fetch and chart wheat and corn futures prices."""
    grains = fetch_yahoo_frame(
        {"ZW=F": "Wheat", "ZC=F": "Corn"},
        start=LONG_START,
    ).dropna()
    summarise(grains, "Grains")
    plot_frame(
        grains,
        title="Grain Futures: Wheat and Corn",
        ylabel="USD cents per bushel",
        lfooter=f"CBOT front-month futures. Data to {latest_date_str(grains)}.",
    )


run_grains()

Grains: 2024-03-20 to 2026-04-17
  Wheat                   n=522  min=495.00  max=700.25
  Corn                    n=522  min=362.00  max=502.00


## Softs and other agricultural commodities

In [15]:
SOFTS = [
    ("ZS=F", "Soybeans", "USD cents per bushel"),
    ("KC=F", "Coffee", "USD cents per pound"),
    ("SB=F", "Sugar", "USD cents per pound"),
    ("CC=F", "Cocoa", "USD per tonne"),
    ("CT=F", "Cotton", "USD cents per pound"),
    ("UFV=F", "Urea", "USD per tonne"),
]

chart_single_tickers(SOFTS, start=LONG_START)

Soybeans: 2024-03-20 to 2026-04-17  min=938.75  max=1248.00


Coffee: 2024-03-20 to 2026-04-17  min=182.40  max=438.90


Sugar: 2024-03-20 to 2026-04-17  min=13.50  max=23.42


Cocoa: 2024-03-20 to 2026-04-17  min=2798.00  max=12565.00


Cotton: 2024-03-20 to 2026-04-17  min=61.06  max=93.41


Urea: 2024-03-20 to 2026-04-16  min=283.00  max=720.25


## ASX indices

In [16]:
def run_asx_indices() -> None:
    """Fetch and chart ASX All Ordinaries and S&P/ASX 200."""
    asx = fetch_yahoo_frame(
        {"^AORD": "All Ordinaries", "^AXJO": "S&P/ASX 200"},
        start=LONG_START,
    )
    summarise(asx, "ASX")
    plot_frame(
        asx,
        title="ASX Indices: All Ordinaries and S&P/ASX 200",
        ylabel="Index",
        lfooter=f"Daily close. Data to {latest_date_str(asx)}.",
    )


run_asx_indices()
chart_single_tickers(
    [("^AXSO", "S&P/ASX Small Ordinaries", "Index")],
    start=LONG_START,
    is_futures=False,
)

ASX: 2024-03-20 to 2026-04-16
  All Ordinaries          n=524  min=7524.30  max=9435.60
  S&P/ASX 200             n=524  min=7343.30  max=9198.60


S&P/ASX Small Ordinaries: 2024-03-20 to 2026-04-16  min=2751.40  max=3992.50


## Watermark

In [17]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-18 10:40:56

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.21
pandas  : 3.0.2
pathlib : 1.0.1
requests: 2.33.1
typing  : 3.10.0.0
yfinance: 1.2.2

Watermark: 2.6.0

